# Example: Generate Training Data from Annotations

In [1]:
from pathlib import Path

from deep_events.csv_to_gaussian import csv_to_gaussian
from deep_events.database.construct import reconstruct_from_folder
from deep_events.database.prepare_yaml import prepare_all_folder
from deep_events.gaussians_to_training import main as gaussian_to_training_events

##### Set the desired parameters

In [ ]:
# The parent data folder(s)
FOLDERS = [Path(".private")]

# The name of the .csv files with the manual annotations
csv_file_pattern = 'labels_1_bf'

# The original data format
img_types = [r'*.ome.tif*']

##### From manual annotations to events ready for downstream operations.
First, we generate the *db.yaml* file - containing all necessary metadata for downstream steps - by reading:
- the metadata in the *ome.tiff* file, 
- the optional general metadata contained in *my_parent_data_folder/db_manual.yaml*,
- and the optional specific metadata contained in any *my_data/db_manual.yaml*.

In [3]:
for folder in FOLDERS:
    prepare_all_folder(Path(folder)) 

tif globs done
globs done
extraction done
Found 1 db files
ome_sources ['.private/220915_mtstaygold_cos7_ZEISS_bf/220915_cos7_mitostaygold_Brightfield_1.ome.tif']
.private/220915_mtstaygold_cos7_ZEISS_bf/220915_cos7_mitostaygold_Brightfield_1.ome.tif


##### Generate the *ground-truth* dataset from the manual annotations.
The dataset is a .tiff file, populated only by gaussians in the locations written in the .csv file

In [4]:
for folder in FOLDERS:
    SIGMA = 10 # the sigma of the gaussian [px]
    csv_to_gaussian(folder, SIGMA, csv_file_pattern)
    
print('\nGround-truth dataset generated.')

.private/220915_mtstaygold_cos7_ZEISS_bf/db.yaml
.private/220915_mtstaygold_cos7_ZEISS_bf/labels_1_bf.csv
.private/220915_mtstaygold_cos7_ZEISS_bf/labels_1_bf.csv
scaling by 1
1098.5783566511182  <- should be >0 and <image_size (2048)
If not, check scale_csv value in db.yaml file.


100%|██████████| 438/438 [01:55<00:00,  3.78it/s]


Done with .private/220915_mtstaygold_cos7_ZEISS_bf/labels_1_bf.csv

Ground-truth dataset generated.


##### Generate the event folder - to be used to populate the database

In [10]:
gt_identifier = "ground_truth_" + csv_file_pattern
settings = {
    'img_identifier': "",
    'gt_identifier': gt_identifier,  
    'db_name': "db.yaml",
    'channel_contrast': "",
    'label': "",
    'add_post_frames': 0,
    'auto_negatives': 0,
}
event_folder = f"event_data_{gt_identifier}"
event_folders = [event_folder]

gaussian_to_training_events(FOLDERS, event_folders, img_types, settings)

print('Event folder generated.')

.private/220915_mtstaygold_cos7_ZEISS_bf/db.yaml
.private/220915_mtstaygold_cos7_ZEISS_bf/db.yaml
tif identifier: *.ome.tif*

.private/220915_mtstaygold_cos7_ZEISS_bf/220915_cos7_mitostaygold_Brightfield_1.ome.tif


100%|██████████| 100/100 [02:01<00:00,  1.21s/it]


ADDITIONAL POST FRAMES 0
[<deep_events.event_extraction.EDA_Event object at 0x7f5ad9b6c050>, <deep_events.event_extraction.EDA_Event object at 0x7f5aec097d90>, <deep_events.event_extraction.EDA_Event object at 0x7f5ad9b5c7d0>, <deep_events.event_extraction.EDA_Event object at 0x7f5ad9f575c0>, <deep_events.event_extraction.EDA_Event object at 0x7f5ad9f57490>, <deep_events.event_extraction.EDA_Event object at 0x7f5ae783d7f0>, <deep_events.event_extraction.EDA_Event object at 0x7f5ad9f84c00>, <deep_events.event_extraction.EDA_Event object at 0x7f5ad9f85040>, <deep_events.event_extraction.EDA_Event object at 0x7f5aec06b050>, <deep_events.event_extraction.EDA_Event object at 0x7f5ae7832a50>, <deep_events.event_extraction.EDA_Event object at 0x7f5b617787d0>, <deep_events.event_extraction.EDA_Event object at 0x7f5ad9f4ed50>, <deep_events.event_extraction.EDA_Event object at 0x7f5ae7843770>, <deep_events.event_extraction.EDA_Event object at 0x7f5ae7843cb0>, <deep_events.event_extraction.EDA_Ev

##### Update the database with the newly generated events, to enable training
Here, we update a MongoDB database with the events later used for training. Please refer to the related section in the README.md file to check how to create it. The database can be later queried to filter training data as desired. The database is deleted (at least for the automatic annotations) every time we run reconstruct_from_folder.

In [11]:
reconstruct_from_folder(
    ".private/event_data_ground_truth_labels_1_bf", # the event folder just created
    "event_data_ground_truth_labels_1_bf" # collection name in the database
)

mongodb://lebpc20.epfl.ch/
.private/event_data_ground_truth_labels_1_bf
.private/event_data_ground_truth_labels_1_bf
15
